# 03 — CalculiX Execution

Preflight the solver, run CalculiX when available, and show the captured solver artifact paths.

In [ ]:
from pathlib import Path
import json
import logging
import os
import shutil
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import src.interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

sample_id = "00689964"
OUTPUT_ROOT = MODULE_ROOT / "outputs"
STATE_B_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "02_fea_constrained_revision"
MESH_SUMMARY_PATH = STATE_B_DIR / "mesh_summary.json"

print("[SETUP] complete")
print(f"  → sample_id : {sample_id}")
print(f"  → state B   : {STATE_B_DIR}")

In [ ]:
def _load_region_summary(payload: dict[str, object]) -> api.RegionSelectionSummary:
    region = payload["region_selection"]
    assert isinstance(region, dict)
    return api.RegionSelectionSummary(
        major_axis=str(region["major_axis"]),
        axis_index=int(region["axis_index"]),
        lower_threshold_mm=float(region["lower_threshold_mm"]),
        upper_threshold_mm=float(region["upper_threshold_mm"]),
        fixed_node_ids=[int(node_id) for node_id in region.get("fixed_node_ids", [])],
        load_node_ids=[int(node_id) for node_id in region.get("load_node_ids", [])],
    )

def _load_mesh_summary(path: Path) -> api.MeshSummary:
    payload = json.loads(path.read_text(encoding="utf-8"))
    return api.MeshSummary(
        inp_path=Path(payload["inp_path"]),
        preview_path=Path(payload["preview_path"]),
        summary_path=Path(payload["summary_path"]),
        node_count=int(payload["node_count"]),
        element_count=int(payload["element_count"]),
        element_type_counts={str(key): int(value) for key, value in payload.get("element_type_counts", {}).items()},
        region_selection=_load_region_summary(payload),
        geometry_step_path=Path(payload["geometry_step_path"]),
        mesh_size_mm=float(payload["mesh_size_mm"]),
    )

In [ ]:
print("[STAGE] solver preflight and run")
config = api.build_baseline_config(
    run_dir=STATE_B_DIR,
    source_step_path=STATE_B_DIR / "fea_revision.step",
    mesh_size_mm=5.0,
    load_magnitude_n=200.0,
)
mesh_summary = _load_mesh_summary(MESH_SUMMARY_PATH)
ccx_path = shutil.which(config.solver_executable)
print(f"  ✓ {config.solver_executable}: {ccx_path}")
if ccx_path and (STATE_B_DIR / "analysis.dat").exists():
    solver_summary = api.SolverRunSummary(
        job_name="analysis",
        run_dir=STATE_B_DIR,
        input_path=mesh_summary.inp_path,
        stdout_path=STATE_B_DIR / "analysis.stdout.txt",
        stderr_path=STATE_B_DIR / "analysis.stderr.txt",
        dat_path=STATE_B_DIR / "analysis.dat",
        frd_path=STATE_B_DIR / "analysis.frd",
        sta_path=STATE_B_DIR / "analysis.sta",
        cvg_path=STATE_B_DIR / "analysis.cvg" if (STATE_B_DIR / "analysis.cvg").exists() else None,
        return_code=0,
        ccx_executable=ccx_path,
        output_files=sorted(path for path in STATE_B_DIR.glob("analysis.*") if path.is_file()),
    )
elif ccx_path:
    solver_summary = api.run_calculix_solver(config, mesh_summary)
else:
    solver_summary = None
    print("  ✗ CalculiX executable not found on PATH; preflight only.")
if solver_summary is not None:
    print(f"  ✓ stdout     : {solver_summary.stdout_path}")
    print(f"  ✓ stderr     : {solver_summary.stderr_path}")
    print(f"  ✓ dat        : {solver_summary.dat_path}")
    print(f"  ✓ frd        : {solver_summary.frd_path}")
    print(f"  ✓ sta        : {solver_summary.sta_path}")
    print(f"  ✓ outputs    : {[path.name for path in solver_summary.output_files]}")
assert mesh_summary.inp_path.exists()
assert mesh_summary.node_count > 0
assert solver_summary is None or solver_summary.return_code == 0

In [ ]:
print("[STAGE] failure case")
mesh_summary = _load_mesh_summary(MESH_SUMMARY_PATH)
bad_mesh_summary = api.MeshSummary(
    inp_path=STATE_B_DIR / "missing.inp",
    preview_path=mesh_summary.preview_path,
    summary_path=mesh_summary.summary_path,
    node_count=mesh_summary.node_count,
    element_count=mesh_summary.element_count,
    element_type_counts=mesh_summary.element_type_counts,
    region_selection=mesh_summary.region_selection,
    geometry_step_path=mesh_summary.geometry_step_path,
    mesh_size_mm=mesh_summary.mesh_size_mm,
)
try:
    api.run_calculix_solver(api.build_baseline_config(run_dir=STATE_B_DIR, source_step_path=STATE_B_DIR / "fea_revision.step"), bad_mesh_summary)
except Exception as exc:
    print(f"  ✓ error type : {type(exc).__name__}")
    print(f"  ✓ message    : {exc}")

## Summary

- The solver preflight is explicit and the missing-`ccx` case is shown clearly.
- When a solver run exists, the notebook surfaces the captured artifacts.
- The failure case shows how malformed solver inputs are reported.